# Complete Adaptive SZ Experiments

Runs to completion (200 rounds each):
- `sz_schedule_b` seed=0 — 3-stage fixed steps (eb 0.001→0.005→0.01)
- `sz_schedule_c` seed=0 — plateau-triggered adaptive schedule
- `sz_schedule_a` seed=1 — additional seed for 2-stage schedule

**First run:** Execute Cell 1, restart runtime, then Cell 2 onward.

**Resuming after timeout:** Mount Drive (Cell 2) and re-run Cell 4 — checkpoints on Drive allow automatic resume from the last completed round.

In [ ]:
%%bash
# Cell 1: install dependencies and patch flwr numpy compatibility
pip install -q 'flwr[simulation]==1.9.0' ray pandas
pip install -q 'protobuf>=4.23' --force-reinstall
pip install -q pysz
# flwr 1.9.0 uses np.float_ which was removed in numpy 2.0
find /usr/local/lib /usr/lib -path '*/flwr/common/typing.py' 2>/dev/null \
    -exec sed -i 's/np\.float_/np.float64/g' {} \; -print
echo ''
echo 'Done. Now: Runtime -> Restart runtime, then run Cell 2.'

In [ ]:
# Cell 2: mount Drive and clone/pull repo
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

REPO_URL  = 'https://github.com/ayananyways/fl-compression-study.git'
CLONE_DIR = '/content/drive/MyDrive/fl-compression-study'

if not os.path.exists(CLONE_DIR):
    subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR], check=True)
    print('Cloned.')
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'pull'], check=True)
    print('Pulled latest.')

os.chdir(CLONE_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3: pre-flight checks
import sys, os

ok = True
try:
    import pysz
    print('pysz OK')
except ImportError:
    print('ERROR: pysz not found — run Cell 1 and restart runtime')
    ok = False

for path in ['fl-flower/run.py', 'fl-flower/adaptive_strategy.py',
             'src/compressors/sz.py']:
    if os.path.exists(path):
        print(f'{path} OK')
    else:
        print(f'ERROR: {path} missing')
        ok = False

if ok:
    print('\nAll checks passed — ready to run.')
else:
    print('\nFix errors above before running Cell 4.')

In [ ]:
# Cell 4: run adaptive experiments
# Checkpoints are saved to Drive after each round.
# If the session times out, re-run this cell after remounting — runs resume automatically.

import subprocess, sys, os, tempfile

OUTPUT = 'results/resnet20_cifar10_adaptive.csv'
os.makedirs('results', exist_ok=True)

CONFIGS = [
    # (schedule, seed, label)
    ('B', 0, 'sz_schedule_b  seed=0  (3-stage: 0.001→0.005→0.01)'),
    ('C', 0, 'sz_schedule_c  seed=0  (plateau-triggered)'),
    ('A', 1, 'sz_schedule_a  seed=1  (additional seed for 2-stage)'),
]

for schedule, seed, label in CONFIGS:
    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')

    cmd = [
        sys.executable, 'fl-flower/run.py',
        '--dataset',      'cifar10',
        '--schedule',     schedule,
        '--seed',         str(seed),
        '--rounds',       '200',
        '--num-clients',  '10',
        '--local-epochs', '2',
        '--output',       OUTPUT,
    ]

    err_file = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False)
    err_path = err_file.name
    err_file.close()

    try:
        subprocess.run(cmd, stderr=open(err_path, 'w'), check=True)
        print(f'\nDone: {label}')
    except subprocess.CalledProcessError as e:
        print(f'\nFAILED (exit {e.returncode}): {label}')
        with open(err_path) as f:
            stderr_text = f.read()
        print('--- stderr (last 4000 chars) ---')
        print(stderr_text[-4000:])
    finally:
        os.unlink(err_path)

print('\n=== All adaptive runs attempted ===')

In [ ]:
# Cell 5: check results
import pandas as pd, os

OUTPUT = 'results/resnet20_cifar10_adaptive.csv'

if not os.path.exists(OUTPUT):
    print('Output file not found.')
else:
    # parse mixed-column format (13-col old rows, 22-col new rows)
    with open(OUTPUT) as f:
        lines = f.readlines()
    header = lines[0].strip().split(',')
    rows = []
    for l in lines[1:]:
        parts = l.strip().split(',')
        if len(parts) == 13:
            rows.append(dict(zip(header, parts)))
        elif len(parts) == 22:
            rows.append({
                'round': parts[1], 'compressor': parts[2], 'seed': parts[3],
                'val_accuracy': parts[6], 'compression_ratio': parts[9],
                'current_eb': parts[21]
            })
    df = pd.DataFrame(rows)
    df['round'] = pd.to_numeric(df['round'], errors='coerce')
    df['val_accuracy'] = pd.to_numeric(df['val_accuracy'], errors='coerce')
    df['seed'] = pd.to_numeric(df['seed'], errors='coerce')

    print(f'Rows in {OUTPUT}: {len(df)}')
    for comp in sorted(df['compressor'].unique()):
        sub = df[df['compressor'] == comp]
        for s in sorted(sub['seed'].dropna().unique()):
            rows2 = sub[sub['seed'] == s]
            max_r = int(rows2['round'].max())
            peak  = rows2['val_accuracy'].max()
            status = 'COMPLETE' if max_r >= 200 else f'PARTIAL ({max_r}/200)'
            print(f'  {comp} s{int(s)}: {status}, peak={peak:.1f}%')